In [1]:
from pystlouisfed import FRED
from dotenv import load_dotenv
import os

import pandas as pd

In [7]:
key = os.getenv("FRED_API_KEY")
fred = FRED(api_key=key)

In [42]:

def getData(name, start, d):

    df = fred.series_observations(name, observation_start= start)

    df['duration'] = d
    
    return df


In [50]:
dataframe = pd.concat([getData('DGS2', '2000-01-01', 2), getData('DGS10', '2000-01-01', 10)])


,realtime_start,realtime_end,value,duration
date,,,,
2000-01-03,2026-10-24,2026-10-24,6.38,2
2000-01-04,2026-10-24,2026-10-24,6.30,2
2000-01-05,2026-10-24,2026-10-24,6.38,2
2000-01-06,2026-10-24,2026-10-24,6.35,2
2000-01-07,2026-10-24,2026-10-24,6.31,2
...,...,...,...,...
2026-10-16,2026-10-24,2026-10-24,5.09,10
2026-10-19,2026-10-24,2026-10-24,5.15,10
2026-10-20,2026-10-24,2026-10-24,5.12,10


In [48]:
def moddur_par(y, T, n=2):
    return (1 - (1 + y / n) ** (-n * T)) / y

In [53]:
dataframe['mod_duration'] = moddur_par(pd.to_numeric(dataframe["value"], errors="coerce") / 100, dataframe['duration'])





In [74]:
wide = (dataframe
        .pivot_table(index="date", columns="duration", values=["value", "mod_duration"])
        .dropna())                                   
wide.columns = [f"{v}_{d}" for v, d in wide.columns]

wide["ratio"] = wide["mod_duration_10"] / wide["mod_duration_2"]   
wide["spread_bp"] = (wide["value_10"] - wide["value_2"]) * 100      

In [75]:
wide

,mod_duration_2,mod_duration_10,value_2,value_10,ratio,spread_bp
date,,,,,,
2000-01-03,1.850136,7.243148,6.38,6.58,3.914928,20.0
2000-01-04,1.851902,7.272999,6.30,6.49,3.927313,19.0
2000-01-05,1.850136,7.229938,6.38,6.62,3.907788,24.0
2000-01-06,1.850798,7.246456,6.35,6.57,3.915315,22.0
2000-01-07,1.851681,7.263028,6.31,6.52,3.922397,21.0
...,...,...,...,...,...,...
2026-10-16,1.886892,7.761550,4.74,5.09,4.113404,35.0
2026-10-19,1.886210,7.739643,4.77,5.15,4.103278,38.0
2026-10-20,1.885755,7.750585,4.79,5.12,4.110071,33.0
